In [1]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 1200 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [2]:
# ============================================================
# TEST ANSWERABILITY GATE
# ============================================================

from answerability import AnswerabilityGate

gate = AnswerabilityGate(
    min_reranker_score=-4.0,
    min_query_coverage=0.20,
)

TEST_CASES = [
    {
        "question": (
            "How does oxygen concentration affect "
            "flame behavior in microgravity?"
        ),
        "description": "Relevant experimental question",
    },
    {
        "question": (
            "What was the flame spread rate in "
            "experiment EXP-0036?"
        ),
        "description": "Specific experiment lookup",
    },
    {
        "question": (
            "What is the history of the Roman Empire?"
        ),
        "description": "Clearly outside the knowledge base",
    },
    {
        "question": (
            "What was the result of experiment EXP-999999?"
        ),
        "description": "Likely missing experiment ID",
    },
]

print("=" * 80)
print("ANSWERABILITY GATE TEST")
print("=" * 80)

for test in TEST_CASES:
    question = test["question"]

    # Reuse the retrieval and reranking components.
    hybrid_results = hybrid_retriever.search(
        query=question,
        dense_top_k=20,
        bm25_top_k=20,
        top_k=20,
    )

    enriched_results = metadata_filter.enrich_results(
        hybrid_results
    )

    reranked_results = reranker.rerank(
        query=question,
        candidates=enriched_results,
        top_k=5,
    )

    result = gate.check(
        question=question,
        candidates=reranked_results,
    )

    print("\n" + "-" * 80)
    print("Test:", test["description"])
    print("Question:", question)
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("Top score:", result["top_score"])
    print(
        "Query coverage:",
        f'{result["query_coverage"]:.2%}'
    )


ANSWERABILITY GATE TEST

CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 143
Reranker Score: -2.8965
Hybrid Rank: 14
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 191
Reranker Score: -3.0640
Hybrid Rank: 20
Dense Rank: None
BM25 Rank: 10

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 151
Reranker Score: -3.0825
Hybrid Rank: 16
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 31
Reranker Score: -3.1579
Hybrid Rank: 4
Dense Rank: None
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 43
Reranker Score: -3.1874
Hybrid Rank: 8
Dense Rank: None
BM25 Rank: 4

-----------------------------

In [3]:
# ============================================================
# VERIFY SPECIFIC EXPERIMENT RETRIEVAL
# ============================================================

question = "What was the flame spread rate in experiment EXP-0036?"

hybrid_results = hybrid_retriever.search(
    query=question,
    dense_top_k=20,
    bm25_top_k=20,
    top_k=20,
)

enriched_results = metadata_filter.enrich_results(
    hybrid_results
)

reranked_results = reranker.rerank(
    query=question,
    candidates=enriched_results,
    top_k=5,
)

print("=" * 80)
print("EXPERIMENT ID VERIFICATION")
print("=" * 80)

for result in reranked_results:
    metadata = result.get("metadata", {})

    print("\nChunk ID:", result.get("chunk_id"))
    print("Experiment ID:", metadata.get("experiment_id"))
    print("Reranker score:", result.get("reranker_score"))

    text = result.get("text", "")
    print("Text contains EXP-0036:", "EXP-0036" in text.upper())

    if "EXP-0036" in text.upper():
        print("Evidence text:")
        print(text[:1500])


CROSS-ENCODER RERANKING
Query: What was the flame spread rate in experiment EXP-0036?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 143
Reranker Score: 9.1453
Hybrid Rank: 1
Dense Rank: 1
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 95
Reranker Score: 8.4560
Hybrid Rank: 12
Dense Rank: None
BM25 Rank: 6

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 142
Reranker Score: 8.3875
Hybrid Rank: 6
Dense Rank: 3
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 151
Reranker Score: 8.3252
Hybrid Rank: 15
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 542
Reranker Score: 8.3070
Hybrid Rank: 13
Dense Rank: 7
BM25 Rank: None
EXPERIMENT ID VERIFICATION

Chunk ID: 143
Experiment ID: EXP-0036
Reranker s

In [5]:

import importlib
import answer.answerability as answerability_module

importlib.reload(answerability_module)

AnswerabilityGate = answerability_module.AnswerabilityGate
gate = AnswerabilityGate()

test_questions = [
    "What was the flame spread rate in experiment EXP-0036?",
    "What was the flame spread rate in experiment EXP-999999?",
    "What was the flame temperature in experiment EXP-0036?",
]

for q in test_questions:
    result = gate.check(q, reranked_results)

    print("Question:", q)
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("-" * 70)


Question: What was the flame spread rate in experiment EXP-0036?
Decision: knowledge_base
Reason: Experiment EXP-0036 contains a numeric value for flame spread rate.
----------------------------------------------------------------------
Question: What was the flame spread rate in experiment EXP-999999?
Decision: llm_fallback
Reason: Experiment EXP-999999 was not found in retrieved evidence.
----------------------------------------------------------------------
Question: What was the flame temperature in experiment EXP-0036?
Decision: llm_fallback
Reason: The requested measurement is not supported by the configured field validator.
----------------------------------------------------------------------
